In [1]:
import pandas as pd

pe_df = pd.read_csv("../Data/cleaned/hcahps_patient_survey_cleaned.csv")
print("Shape:", pe_df.shape)
print("\nColumns:")
for col in pe_df.columns:
    print(f"  {col}")

Shape: (325652, 18)

Columns:
  facility_id
  facility_name
  address
  citytown
  state
  zip_code
  countyparish
  telephone_number
  hcahps_measure_id
  hcahps_question
  hcahps_answer_description
  patient_survey_star_rating
  hcahps_answer_percent
  hcahps_linear_mean_value
  number_of_completed_surveys
  survey_response_rate_percent
  start_date
  end_date


In [3]:
print("unique measures:", pe_df["hcahps_measure_id"].nunique())
print("\nsample measures:")
print(pe_df[["hcahps_measure_id", "hcahps_question"]].drop_duplicates().to_string())

unique measures: 68

sample measures:
            hcahps_measure_id                                                                                                                     hcahps_question
0                H_COMP_1_A_P                                                                  Patients who reported that their nurses "Always" communicated well
1               H_COMP_1_SN_P                                                    Patients who reported that their nurses "Sometimes" or "Never" communicated well
2                H_COMP_1_U_P                                                                 Patients who reported that their nurses "Usually" communicated well
3       H_COMP_1_LINEAR_SCORE                                                                                             Nurse communication - linear mean score
4        H_COMP_1_STAR_RATING                                                                                                   Nurse communication - st

In [4]:
summary_measures = pe_df[
    pe_df["hcahps_measure_id"].str.contains("STAR_RATING|LINEAR_SCORE|H_STAR_RATING")
]

print("rows after filter:", len(summary_measures))
print("measures kept:")
print(summary_measures["hcahps_measure_id"].unique())

rows after filter: 81413
measures kept:
['H_COMP_1_LINEAR_SCORE' 'H_COMP_1_STAR_RATING' 'H_COMP_2_LINEAR_SCORE'
 'H_COMP_2_STAR_RATING' 'H_COMP_5_LINEAR_SCORE' 'H_COMP_5_STAR_RATING'
 'H_COMP_6_LINEAR_SCORE' 'H_COMP_6_STAR_RATING' 'H_CLEAN_LINEAR_SCORE'
 'H_CLEAN_STAR_RATING' 'H_QUIET_LINEAR_SCORE' 'H_QUIET_STAR_RATING'
 'H_HSP_RATING_LINEAR_SCORE' 'H_HSP_RATING_STAR_RATING'
 'H_RECMND_LINEAR_SCORE' 'H_RECMND_STAR_RATING' 'H_STAR_RATING']


In [6]:
summary_measures = summary_measures.copy()

for col in ["patient_survey_star_rating", "hcahps_answer_percent", "hcahps_linear_mean_value"]:
    summary_measures[col] = summary_measures[col].replace({"Not Available": None, "Not Applicable": None})
    summary_measures[col] = pd.to_numeric(summary_measures[col], errors="coerce")

linear_pivot = summary_measures[summary_measures["hcahps_measure_id"].str.contains("LINEAR|H_STAR_RATING")].pivot_table(
    index="facility_id",
    columns="hcahps_measure_id",
    values="hcahps_linear_mean_value",
    aggfunc="first"
).reset_index()
linear_pivot.columns.name = None

star_pivot = summary_measures[summary_measures["hcahps_measure_id"].str.contains("STAR")].pivot_table(
    index="facility_id",
    columns="hcahps_measure_id",
    values="patient_survey_star_rating",
    aggfunc="first"
).reset_index()
star_pivot.columns.name = None

hospital_info = summary_measures[["facility_id", "facility_name", "address",
                                   "citytown", "state", "zip_code",
                                   "countyparish", "telephone_number"]].drop_duplicates("facility_id")

df_final = hospital_info.merge(linear_pivot, on="facility_id", how="outer")
df_final = df_final.merge(star_pivot, on="facility_id", how="outer")

print("final shape:", df_final.shape)
print("duplicate facility_ids:", df_final["facility_id"].duplicated().sum())
df_final.head(2)

final shape: (4789, 25)
duplicate facility_ids: 0


,facility_id,facility_name,address,citytown,state,zip_code,countyparish,telephone_number,H_CLEAN_LINEAR_SCORE,H_COMP_1_LINEAR_SCORE,...,H_RECMND_LINEAR_SCORE,H_CLEAN_STAR_RATING,H_COMP_1_STAR_RATING,H_COMP_2_STAR_RATING,H_COMP_5_STAR_RATING,H_COMP_6_STAR_RATING,H_HSP_RATING_STAR_RATING,H_QUIET_STAR_RATING,H_RECMND_STAR_RATING,H_STAR_RATING
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,86.0,90.0,...,91.0,3.0,3.0,4.0,3.0,4.0,4.0,4.0,5.0,4.0
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,84.0,91.0,...,85.0,3.0,3.0,4.0,2.0,3.0,3.0,4.0,3.0,3.0


In [7]:
df_final.to_csv("../Data/cleaned/hcahps_patient_survey_cleaned.csv", index=False)
print("hcahps_patient_survey saved")

hcahps_patient_survey saved
